In [2]:

import pandas as pd
import os

# corupted  dataset 
corrupt_df = pd.read_csv("../data/corrupted_synthetic_dataset.csv")

#  PakWheels datase
pak_df = pd.read_csv("../data/pakwheels.csv")

print("Corrupted dataset shape:", corrupt_df.shape)
print("PakWheels dataset shape:", pak_df.shape)

Corrupted dataset shape: (101980, 9)
PakWheels dataset shape: (62302, 14)


In [ ]:
print(" CLEANING CORRUPTED SYNTHETIC DATASET ===")
print("Before cleaning:", corrupt_df.shape)

# 1. Remove duplicate rows
corrupt_df = corrupt_df.drop_duplicates(subset=["sale_id"])
print("After removing duplicates:", corrupt_df.shape)

# 2. Remove rows where customer_id =null
corrupt_df = corrupt_df.dropna(subset=["customer_id"])
print("After dropping null customer_id:", corrupt_df.shape)

# 3.  remove rows where quantity is less than 1
corrupt_df = corrupt_df[corrupt_df["quantity"] >= 1]
print("After removing invalid quantity:", corrupt_df.shape)

# 4. remove rows where amount exceeds 5000
corrupt_df = corrupt_df[corrupt_df["amount"] <= 5000]
print("After removing invalid amount:", corrupt_df.shape)


valid_channels = ["online", "store", "app"]
corrupt_df = corrupt_df[corrupt_df["channel"].isin(valid_channels)]
print("After removing invalid channel:", corrupt_df.shape)

print("\nCleaning complete!")

 CLEANING CORRUPTED SYNTHETIC DATASET ===
Before cleaning: (101980, 9)
After removing duplicates: (100000, 9)
After dropping null customer_id: (99033, 9)
After removing invalid quantity: (98530, 9)
After removing invalid amount: (98055, 9)
After removing invalid channel: (97129, 9)

Cleaning complete!


In [4]:
os.makedirs("../data", exist_ok=True)
corrupt_df.to_csv("../data/cleaned_synthetic_dataset.csv", index=False)
print("Saved -> data/cleaned_synthetic_dataset.csv")

Saved -> data/cleaned_synthetic_dataset.csv


In [5]:
print("=== CLEANING PAKWHEELS DATASET ===")
print("Before cleaning:", pak_df.shape)

# 1. remove rows milage low 100 or above 500,000
pak_df = pak_df[(pak_df["mileage"] >= 100) & (pak_df["mileage"] <= 500000)]
print("After fixing mileage:", pak_df.shape)

# 2. Fix engine
pak_df = pak_df[
    (pak_df["engine"].isna()) |
    ((pak_df["engine"] >= 660) & (pak_df["engine"] <= 5000))
]
print("After fixing engine:", pak_df.shape)

# 3. Fix price 
pak_df = pak_df[
    (pak_df["price"].isna()) |
    (pak_df["price"] <= 50000000)
]
print("After fixing price:", pak_df.shape)


pak_df["assembly"] = pak_df["assembly"].fillna("Local")
print("After filling assembly nulls:", pak_df.shape)

print("\nCleaning complete!")

=== CLEANING PAKWHEELS DATASET ===
Before cleaning: (62302, 14)
After fixing mileage: (59849, 14)
After fixing engine: (59426, 14)
After fixing price: (59338, 14)
After filling assembly nulls: (59338, 14)

Cleaning complete!


In [6]:
pak_df.to_csv("../data/cleaned_pakwheels_dataset.csv", index=False)
print("Saved -> data/cleaned_pakwheels_dataset.csv")

Saved -> data/cleaned_pakwheels_dataset.csv


In [7]:
import great_expectations as gx
from great_expectations.core.expectation_suite import ExpectationSuite
from great_expectations.expectations import (
    ExpectColumnValuesToNotBeNull,
    ExpectColumnValuesToBeUnique,
    ExpectColumnValuesToBeBetween,
    ExpectColumnValuesToBeInSet,
    ExpectColumnValuesToBeOfType,
    ExpectTableRowCountToBeBetween,
)

context = gx.get_context()

# Rebuild the same suite from Task 2
suite_synthetic = ExpectationSuite(name="cleaned_synthetic_suite")
suite_synthetic.add_expectation(ExpectColumnValuesToNotBeNull(column="sale_id"))
suite_synthetic.add_expectation(ExpectColumnValuesToNotBeNull(column="customer_id"))
suite_synthetic.add_expectation(ExpectColumnValuesToNotBeNull(column="product_id"))
suite_synthetic.add_expectation(ExpectColumnValuesToNotBeNull(column="sale_ts"))
suite_synthetic.add_expectation(ExpectColumnValuesToNotBeNull(column="quantity"))
suite_synthetic.add_expectation(ExpectColumnValuesToNotBeNull(column="amount"))
suite_synthetic.add_expectation(ExpectColumnValuesToNotBeNull(column="channel"))
suite_synthetic.add_expectation(ExpectColumnValuesToNotBeNull(column="currency"))
suite_synthetic.add_expectation(ExpectColumnValuesToBeUnique(column="sale_id"))
suite_synthetic.add_expectation(ExpectColumnValuesToBeBetween(column="quantity", min_value=1, max_value=10))
suite_synthetic.add_expectation(ExpectColumnValuesToBeBetween(column="amount", min_value=0.0, max_value=5000.0))
suite_synthetic.add_expectation(ExpectColumnValuesToBeInSet(column="channel", value_set=["online", "store", "app"]))
suite_synthetic.add_expectation(ExpectColumnValuesToBeInSet(column="currency", value_set=["USD", "EUR", "PKR"]))
suite_synthetic.add_expectation(ExpectColumnValuesToBeOfType(column="quantity", type_="int64"))
suite_synthetic.add_expectation(ExpectColumnValuesToBeOfType(column="amount", type_="float64"))
suite_synthetic.add_expectation(ExpectTableRowCountToBeBetween(min_value=90000, max_value=200000))

# Load cleaned file
cleaned_synthetic = pd.read_csv("../data/cleaned_synthetic_dataset.csv")

ds = context.data_sources.add_or_update_pandas(name="cleaned_synthetic")
asset = ds.add_dataframe_asset(name="cleaned_synthetic_asset")
batch_def = asset.add_batch_definition_whole_dataframe("cleaned_synthetic_batch")
batch = batch_def.get_batch(batch_parameters={"dataframe": cleaned_synthetic})

results_synthetic = batch.validate(suite_synthetic)

total  = len(results_synthetic.results)
passed = sum(1 for r in results_synthetic.results if r.success)
failed = total - passed

print("=== CLEANED SYNTHETIC DATASET ===")
print(f"Total  : {total}")
print(f"Passed : {passed} ")
print(f"Failed : {failed} ")
print(f"Overall: {'PASSED ' if results_synthetic.success else 'FAILED'}")

Calculating Metrics:   0%|          | 0/75 [00:00<?, ?it/s]

=== CLEANED SYNTHETIC DATASET ===
Total  : 16
Passed : 16 
Failed : 0 
Overall: PASSED 


In [11]:
# Rebuild the same suite from Task 3
suite_pak = ExpectationSuite(name="cleaned_pakwheels_suite")
suite_pak.add_expectation(ExpectColumnValuesToNotBeNull(column="addref"))
suite_pak.add_expectation(ExpectColumnValuesToNotBeNull(column="city"))
suite_pak.add_expectation(ExpectColumnValuesToNotBeNull(column="make"))
suite_pak.add_expectation(ExpectColumnValuesToNotBeNull(column="model"))
suite_pak.add_expectation(ExpectColumnValuesToNotBeNull(column="transmission"))
suite_pak.add_expectation(ExpectColumnValuesToNotBeNull(column="registered"))
suite_pak.add_expectation(ExpectColumnValuesToNotBeNull(column="mileage"))
suite_pak.add_expectation(ExpectColumnValuesToBeUnique(column="addref"))
suite_pak.add_expectation(ExpectColumnValuesToBeBetween(column="year", min_value=1990, max_value=2023))
suite_pak.add_expectation(ExpectColumnValuesToBeBetween(column="engine", min_value=660, max_value=5000))
suite_pak.add_expectation(ExpectColumnValuesToBeBetween(column="mileage", min_value=100, max_value=500000))
suite_pak.add_expectation(ExpectColumnValuesToBeBetween(column="price", min_value=100000, max_value=50000000))
suite_pak.add_expectation(ExpectColumnValuesToBeInSet(column="transmission", value_set=["Automatic", "Manual"]))
suite_pak.add_expectation(ExpectColumnValuesToBeInSet(column="fuel", value_set=["Petrol", "Diesel", "Hybrid", "CNG", "Electric", "LPG"]))
suite_pak.add_expectation(ExpectColumnValuesToBeInSet(column="assembly", value_set=["Imported", "Local"]))
suite_pak.add_expectation(ExpectColumnValuesToBeOfType(column="mileage", type_="int64"))
suite_pak.add_expectation(ExpectColumnValuesToBeOfType(column="engine", type_="float64"))
suite_pak.add_expectation(ExpectTableRowCountToBeBetween(min_value=55000, max_value=100000))

# Load cleaned file
cleaned_pak = pd.read_csv("../data/cleaned_pakwheels_dataset.csv")

ds2 = context.data_sources.add_or_update_pandas(name="cleaned_pakwheels")
asset2 = ds2.add_dataframe_asset(name="cleaned_pakwheels_asset")
batch_def2 = asset2.add_batch_definition_whole_dataframe("cleaned_pakwheels_batch")
batch2 = batch_def2.get_batch(batch_parameters={"dataframe": cleaned_pak})

results_pak = batch2.validate(suite_pak)

total2  = len(results_pak.results)
passed2 = sum(1 for r in results_pak.results if r.success)
failed2 = total2 - passed2

print("=== CLEANED PAKWHEELS DATASET ===")
print(f"Total  : {total2}")
print(f"Passed : {passed2} ")
print(f"Failed : {failed2} ")
print(f"Overall: {'PASSED ' if results_pak.success else 'FAILED '}")

Calculating Metrics:   0%|          | 0/93 [00:00<?, ?it/s]

=== CLEANED PAKWHEELS DATASET ===
Total  : 18
Passed : 18 
Failed : 0 
Overall: PASSED 


In [12]:
os.makedirs("../reports", exist_ok=True)

def save_report(results, filepath, label):
    with open(filepath, "w", encoding="utf-8") as f:
        total  = len(results.results)
        passed = sum(1 for r in results.results if r.success)
        failed = total - passed
        f.write("="*55 + "\n")
        f.write(f"  DATA QUALITY REPORT (AFTER CLEANING) — {label}\n")
        f.write("="*55 + "\n\n")
        f.write(f"Overall Result : {'PASSED' if results.success else 'FAILED'}\n")
        f.write(f"Total Checks   : {total}\n")
        f.write(f"Passed         : {passed}\n")
        f.write(f"Failed         : {failed}\n\n")
        f.write("─"*55 + "\n")
        f.write("  DETAILED RESULTS\n")
        f.write("─"*55 + "\n\n")
        for r in results.results:
            status   = "PASS" if r.success else "FAIL"
            exp_type = r.expectation_config.type
            col      = r.expectation_config.kwargs.get("column", "TABLE-LEVEL")
            f.write(f"[{status}]  Column: {col}\n")
            f.write(f"         Check : {exp_type}\n")
            if not r.success and r.result:
                uc = r.result.get("unexpected_count", "N/A")
                up = r.result.get("unexpected_percent", 0)
                if uc != "N/A":
                    f.write(f"         Bad rows  : {uc}\n")
                    f.write(f"         Bad %     : {round(up,2)}%\n")
            f.write("\n")
    print(f"Saved → {filepath}")

save_report(results_synthetic, "../reports/cleaned_synthetic_report.txt",  "SYNTHETIC DATASET")
save_report(results_pak,       "../reports/cleaned_pakwheels_report.txt",   "PAKWHEELS DATASET")

Saved → ../reports/cleaned_synthetic_report.txt
Saved → ../reports/cleaned_pakwheels_report.txt
